# Data Processing: AST-based Feature Extraction for QNN

## Mục tiêu

Notebook này chuyển đổi file JSONL kiểu Vul4J (mỗi dòng: `{vul_id, file, method, version, label, code}`) thành dataset dạng **fixed-size vector + label** để train QNN (Quantum Neural Network).

## Tại sao cần bước này?

- **QNN không nhận trực tiếp đồ thị AST** dạng node/edge list như GNN
- **QNN cần đầu vào là vector số có kích thước cố định** ($\mathbb{R}^d$) để mã hoá lên n qubit (angle/amplitude encoding)
- Giải pháp: Parse Java code → AST → trích xuất đặc trưng cấu trúc → vector cố định

## Phương pháp

1. **Parse Java snippet** → AST bằng tree-sitter
2. **AST → fixed-size vector** bằng Feature Hashing:
   - Histogram node types (IfStatement, method_invocation, ...)
   - Histogram edges (parent→child theo loại node)
   - Histogram root-to-leaf paths (motif cấu trúc)
   - Scalar stats (n_nodes, max_depth, avg_branching, ...)
3. **Xuất CSV hoàn chỉnh** với tất cả samples để sử dụng linh hoạt

## Output

- `qnn_features.csv`: File CSV chứa vectors, labels và metadata
- `summary.json`: Thống kê dataset
- `failed_parse.jsonl`: Các samples parse lỗi (nếu có)

## 1. Import Dependencies

In [24]:
from __future__ import annotations

import dataclasses
import hashlib
import json
import math
import os
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

print("✓ Dependencies imported successfully")

✓ Dependencies imported successfully


## 2. Utility Functions

Các hàm tiện ích cho hashing, normalization và đọc JSONL.

In [25]:
def stable_hash_to_bucket(text: str, dim: int, seed: int = 0) -> int:
    """
    Hash ổn định (không phụ thuộc Python hash randomization), 
    để feature hashing reproducible.
    """
    h = hashlib.md5((str(seed) + "::" + text).encode("utf-8")).hexdigest()
    return int(h, 16) % dim


def iter_jsonl(path: str) -> Iterable[Dict[str, Any]]:
    """Đọc file JSONL từng dòng."""
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON decode error at line {line_no}: {e}") from e


def normalize_newlines(code: str) -> str:
    """Chuẩn hoá newlines để parser/tokenizer nhất quán."""
    return code.replace("\r\n", "\n").replace("\r", "\n")


print("✓ Utility functions defined")

✓ Utility functions defined


## 3. Java AST Parser với Tree-sitter

### Tại sao dùng tree-sitter?

- Parse nhanh và chịu lỗi tốt
- Không cần Java compiler
- Xuất AST node-type rõ ràng

In [ ]:
class JavaASTParser:
    """
    Wrapper cho tree-sitter Java parser.
    Hỗ trợ parse snippet Java code (method/class fragments).
    """

    def __init__(self):
        try:
            import tree_sitter
            from tree_sitter_languages import get_language
            
            # Get Java language
            java_lang = get_language('java')
            
            # Create parser and set language (tree-sitter 0.20.4 API)
            self._parser = tree_sitter.Parser()
            self._parser.set_language(java_lang)
            
        except Exception as e:
            raise RuntimeError(
                f"Không thể khởi tạo tree-sitter parser. Lỗi: {e}. "
                "Đảm bảo đã cài: pip install tree-sitter==0.20.4 tree-sitter-languages==1.10.2"
            ) from e

    @staticmethod
    def _wrap_as_class(snippet: str) -> str:
        """Wrap snippet vào class giả để parse."""
        return "class Dummy {\n" + snippet + "\n}\n"

    @staticmethod
    def _wrap_as_method_body(snippet: str) -> str:
        """Wrap snippet vào method body giả."""
        return "class Dummy {\nvoid __dummy__() {\n" + snippet + "\n}\n}\n"

    @staticmethod
    def _count_error_nodes(root) -> int:
        """Đếm số node ERROR trong AST."""
        count = 0
        stack = [root]
        while stack:
            n = stack.pop()
            if n.type == "ERROR":
                count += 1
            stack.extend(reversed(n.children))
        return count

    def parse(self, snippet: str):
        """
        Parse snippet và trả về AST root node.
        Thử 2 chiến lược wrap để tối ưu parse success.
        """
        src = self._wrap_as_class(snippet)
        tree = self._parser.parse(bytes(src, "utf-8"))
        root = tree.root_node
        err = self._count_error_nodes(root)

        if err > 0:
            src2 = self._wrap_as_method_body(snippet)
            tree2 = self._parser.parse(bytes(src2, "utf-8"))
            root2 = tree2.root_node
            err2 = self._count_error_nodes(root2)

            if err2 < err:
                return root2, err2
        return root, err


print("✓ JavaASTParser class defined")

✓ JavaASTParser class defined


## 4. AST Stats Dataclass

In [27]:
@dataclasses.dataclass
class ASTStats:
    """Thống kê cấu trúc AST."""
    n_nodes: int = 0
    n_edges: int = 0
    n_leaves: int = 0
    max_depth: int = 0
    avg_branching: float = 0.0
    truncated: int = 0
    error_nodes: int = 0


print("✓ ASTStats dataclass defined")

✓ ASTStats dataclass defined


## 5. Feature Extraction: AST → Fixed-size Vector

### Tại sao Feature Hashing?

- Giữ vector có kích thước cố định $d$ (phù hợp QNN)
- Không cần xây vocab node-types toàn cục
- Ổn định và dễ mở rộng

### Thành phần feature:

1. **Node type counts**: `bucket("NT:<node_type>") += 1`
2. **Edge type counts**: `bucket("E:<parent_type>-><child_type>") += 1`
3. **Root-to-leaf paths**: `bucket("P:<t1>/<t2>/...") += 1`
4. **Scalar stats**: n_nodes, n_edges, max_depth, ... (các chiều cuối)

In [28]:
class ASTHasherFeaturizer:
    """
    Biến AST thành vector R^dim bằng feature hashing.
    """

    def __init__(
        self,
        dim: int = 128,
        seed: int = 0,
        max_nodes: int = 4000,
        max_path_len: int = 8,
        scalar_dims: int = 8,
        use_edges: bool = True,
        use_paths: bool = True,
    ):
        if dim <= scalar_dims:
            raise ValueError("dim phải > scalar_dims để còn chỗ cho hashed features.")
        self.dim = dim
        self.hashed_dim = dim - scalar_dims
        self.scalar_dims = scalar_dims
        self.seed = seed
        self.max_nodes = max_nodes
        self.max_path_len = max_path_len
        self.use_edges = use_edges
        self.use_paths = use_paths

        self._parser = JavaASTParser()

    def featurize(self, code: str) -> Tuple[np.ndarray, ASTStats]:
        """Parse code và trích xuất feature vector."""
        code = normalize_newlines(code)
        root, err_nodes = self._parser.parse(code)

        # Duyệt AST
        node_types: List[str] = []
        parent: List[int] = []
        depth: List[int] = []
        child_count: List[int] = []

        stack = [(root, -1, 0)]
        truncated = 0

        while stack:
            n, p, d = stack.pop()
            if len(node_types) >= self.max_nodes:
                truncated = 1
                break

            nid = len(node_types)
            node_types.append(n.type)
            parent.append(p)
            depth.append(d)
            child_count.append(len(n.children))

            for ch in reversed(n.children):
                stack.append((ch, nid, d + 1))

        n_nodes = len(node_types)
        n_edges = sum(1 for p in parent if p != -1)
        n_leaves = sum(1 for c in child_count[:n_nodes] if c == 0)
        max_depth = max(depth) if depth else 0
        avg_branching = float(sum(child_count[:n_nodes]) / max(1, n_nodes))

        stats = ASTStats(
            n_nodes=n_nodes,
            n_edges=n_edges,
            n_leaves=n_leaves,
            max_depth=max_depth,
            avg_branching=avg_branching,
            truncated=truncated,
            error_nodes=err_nodes,
        )

        x = np.zeros((self.dim,), dtype=np.float32)

        # 1) Node type histogram (hashed)
        for t in node_types:
            b = stable_hash_to_bucket("NT:" + t, self.hashed_dim, self.seed)
            x[b] += 1.0

        # 2) Edge type histogram (parent->child)
        if self.use_edges:
            for i in range(n_nodes):
                p = parent[i]
                if p == -1:
                    continue
                et = f"E:{node_types[p]}->{node_types[i]}"
                b = stable_hash_to_bucket(et, self.hashed_dim, self.seed)
                x[b] += 1.0

        # 3) Root-to-leaf path histogram
        if self.use_paths and n_nodes > 0:
            for leaf_id in range(n_nodes):
                if child_count[leaf_id] != 0:
                    continue
                path_types = []
                cur = leaf_id
                steps = 0
                while cur != -1 and steps < self.max_path_len:
                    path_types.append(node_types[cur])
                    cur = parent[cur]
                    steps += 1
                path_types.reverse()
                if not path_types:
                    continue
                key = "P:" + "/".join(path_types)
                b = stable_hash_to_bucket(key, self.hashed_dim, self.seed)
                x[b] += 1.0

        # 4) Scalar stats
        scalars = [
            math.log1p(stats.n_nodes),
            math.log1p(stats.n_edges),
            math.log1p(stats.n_leaves),
            float(stats.max_depth),
            float(stats.avg_branching),
            float(stats.truncated),
            float(stats.error_nodes),
            1.0,  # bias constant
        ]
        scalars = (scalars + [0.0] * self.scalar_dims)[: self.scalar_dims]
        x[self.hashed_dim :] = np.asarray(scalars, dtype=np.float32)

        return x, stats


print("✓ ASTHasherFeaturizer class defined")

✓ ASTHasherFeaturizer class defined


## 6. Data Record Class

In [29]:
@dataclasses.dataclass
class Record:
    """Data record từ JSONL."""
    rid: str
    vul_id: str
    file: str
    method: str
    version: str
    label: int
    code: str

    @staticmethod
    def from_json(obj: Dict[str, Any]) -> "Record":
        vul_id = str(obj.get("vul_id", ""))
        file = str(obj.get("file", ""))
        method = str(obj.get("method", ""))
        version = str(obj.get("version", ""))
        label = int(obj.get("label", 0))
        code = str(obj.get("code", ""))

        rid = f"{vul_id}::{file}::{method}::{version}"
        return Record(
            rid=rid, vul_id=vul_id, file=file, 
            method=method, version=version, label=label, code=code
        )


print("✓ Record dataclass defined")

✓ Record dataclass defined


## 7. Deduplication

In [30]:
def deduplicate_records(records: List[Record]) -> List[Record]:
    """
    Loại trùng để giảm bias do cùng một snippet lặp lại nhiều lần.
    """
    seen = set()
    out = []
    for r in records:
        key = hashlib.md5(
            (r.vul_id + "\n" + r.file + "\n" + r.method + "\n" + 
             r.version + "\n" + str(r.label) + "\n" + 
             normalize_newlines(r.code)).encode("utf-8")
        ).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out


print("✓ Deduplication function defined")

✓ Deduplication function defined


## 8. Configuration

Cấu hình parameters cho feature extraction.

In [31]:
# ==================== CONFIGURATION ====================
INPUT_FILE = "dataoutput_dataset.jsonl"
OUTPUT_DIR = "qnn_dataset"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "qnn_features.csv")

# Feature parameters
DIM = 128                # Kích thước vector đặc trưng
SEED = 42               # Random seed
MAX_NODES = 4000        # Giới hạn số nodes trong AST
MAX_PATH_LEN = 8        # Độ dài tối đa của root-to-leaf path
SCALAR_DIMS = 8         # Số scalar stats ở cuối vector
USE_EDGES = True        # Sử dụng edge type features
USE_PATHS = True        # Sử dụng path features
DEDUP = True            # Loại bỏ duplicates

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✓ Configuration set")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_CSV}")
print(f"  Feature dim: {DIM}")

✓ Configuration set
  Input: dataoutput_dataset.jsonl
  Output: qnn_dataset\qnn_features.csv
  Feature dim: 128


## 9. Load Data from JSONL

In [32]:
# Load records
print("Loading data from JSONL...")
records = [Record.from_json(obj) for obj in iter_jsonl(INPUT_FILE)]
print(f"  Loaded {len(records)} records")

# Deduplicate
if DEDUP:
    before = len(records)
    records = deduplicate_records(records)
    after = len(records)
    print(f"  After deduplication: {after} records (removed {before - after})")

if not records:
    raise RuntimeError("No records loaded!")

Loading data from JSONL...
  Loaded 1865 records
  After deduplication: 1578 records (removed 287)


## 10. Feature Extraction & DataFrame Creation

Process tất cả samples và tạo DataFrame với features.

In [33]:
# Initialize featurizer
print("\nInitializing featurizer...")
featurizer = ASTHasherFeaturizer(
    dim=DIM,
    seed=SEED,
    max_nodes=MAX_NODES,
    max_path_len=MAX_PATH_LEN,
    scalar_dims=SCALAR_DIMS,
    use_edges=USE_EDGES,
    use_paths=USE_PATHS,
 )
print("  ✓ Featurizer ready")

# Process all records
print(f"\nProcessing {len(records)} records...")
failed_parse_path = os.path.join(OUTPUT_DIR, "failed_parse.jsonl")
failed_f = open(failed_parse_path, "w", encoding="utf-8")

data_rows = []
stats_agg = {
    "n": 0,
    "pos": 0,
    "neg": 0,
    "avg_nodes": 0.0,
    "avg_edges": 0.0,
    "avg_error_nodes": 0.0,
    "truncated": 0,
}

for i, r in enumerate(records):
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(records)}...")
    
    try:
        vec, st = featurizer.featurize(r.code)
    except Exception as e:
        # Log failed parse
        failed_f.write(
            json.dumps({
                "id": r.rid,
                "vul_id": r.vul_id,
                "error": str(e),
                "code": r.code
            }, ensure_ascii=False) + "\n"
)
        vec = np.zeros((DIM,), dtype=np.float32)
        st = ASTStats(truncated=1, error_nodes=999999)
    
    # Create row with metadata + features
    row = {
        "id": r.rid,
        "vul_id": r.vul_id,
        "file": r.file,
        "method": r.method,
        "version": r.version,
        "label": int(r.label),
        "ast_n_nodes": st.n_nodes,
        "ast_n_edges": st.n_edges,
        "ast_n_leaves": st.n_leaves,
        "ast_max_depth": st.max_depth,
        "ast_avg_branching": st.avg_branching,
        "ast_error_nodes": st.error_nodes,
        "ast_truncated": st.truncated,
    }
    
    # Add feature vector columns
    for j in range(DIM):
        row[f"feat_{j}"] = float(vec[j])
    
    data_rows.append(row)
    
    # Update stats
    stats_agg["n"] += 1
    if r.label == 1:
        stats_agg["pos"] += 1
    else:
        stats_agg["neg"] += 1
    stats_agg["avg_nodes"] += st.n_nodes
    stats_agg["avg_edges"] += st.n_edges
    stats_agg["avg_error_nodes"] += st.error_nodes
    stats_agg["truncated"] += st.truncated

failed_f.close()

if stats_agg["n"] > 0:
    stats_agg["avg_nodes"] /= stats_agg["n"]
    stats_agg["avg_edges"] /= stats_agg["n"]
    stats_agg["avg_error_nodes"] /= stats_agg["n"]

print(f"\n✓ Processed all {len(records)} records")
print(f"  Positive samples: {stats_agg['pos']}")
print(f"  Negative samples: {stats_agg['neg']}")
print(f"  Avg AST nodes: {stats_agg['avg_nodes']:.1f}")
print(f"  Avg error nodes: {stats_agg['avg_error_nodes']:.2f}")


Initializing featurizer...


RuntimeError: Thiếu dependency tree-sitter. Lỗi: Không thể khởi tạo tree-sitter parser. Lỗi 1: __init__() takes exactly 1 argument (2 given), Lỗi 2: __init__() takes exactly 1 argument (2 given). Hãy thử: pip uninstall tree-sitter tree-sitter-languages && pip install tree-sitter==0.20.4 tree-sitter-languages==1.10.2. Cài: pip install tree-sitter tree-sitter-languages

## 11. Save to CSV

In [ ]:
# Create DataFrame
print("\nCreating DataFrame...")
df = pd.DataFrame(data_rows)

# Save to CSV
print(f"Saving to {OUTPUT_CSV}...")
df.to_csv(OUTPUT_CSV, index=False)
print(f"  ✓ Saved {len(df)} rows to CSV")

# Display first few rows
print("\nFirst 3 rows (metadata columns only):")
display(df[["id", "vul_id", "label", "ast_n_nodes", "ast_error_nodes"]].head(3))

print(f"\nDataFrame shape: {df.shape}")
print(f"  Metadata columns: 13")
print(f"  Feature columns: {DIM}")

## 12. Save Summary Statistics

In [ ]:
# Create summary
summary = {
    "input": os.path.abspath(INPUT_FILE),
    "output_csv": os.path.abspath(OUTPUT_CSV),
    "output_dir": os.path.abspath(OUTPUT_DIR),
    "config": {
        "dim": DIM,
        "seed": SEED,
        "max_nodes": MAX_NODES,
        "max_path_len": MAX_PATH_LEN,
        "scalar_dims": SCALAR_DIMS,
        "use_edges": USE_EDGES,
        "use_paths": USE_PATHS,
        "dedup": DEDUP,
    },
    "statistics": stats_agg,
    "label_distribution": {
        "positive": int(stats_agg["pos"]),
        "negative": int(stats_agg["neg"]),
        "total": int(stats_agg["n"]),
        "positive_ratio": float(stats_agg["pos"] / stats_agg["n"]) if stats_agg["n"] > 0 else 0.0,
    },
    "notes": [
        "Features are AST-structural counts hashed into fixed dimension.",
        "CSV contains all samples - you can split train/val/test as needed.",
        "Consider normalizing features (log1p + z-score) before feeding to QNN.",
        "For QNN with n_qubits features, consider reducing dim or using PCA.",
    ],
}

# Save summary
summary_path = os.path.join(OUTPUT_DIR, "summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n✓ Summary saved to {summary_path}")
print("\n" + "="*60)
print("🎉 DATA PROCESSING COMPLETE!")
print("="*60)
print(f"\nOutput files:")
print(f"  1. {OUTPUT_CSV} - Main dataset")
print(f"  2. {summary_path} - Statistics")
print(f"  3. {failed_parse_path} - Failed parses")
print(f"\nNext steps:")
print(f"  - Split CSV into train/val/test as needed")
print(f"  - Normalize features before QNN training")
print(f"  - Consider PCA if dim >> n_qubits")